In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

plt.style.use("seaborn-v0_8-paper")  # 干净的浅色主题（可换 "default"）

mpl.rcParams.update({
    # ---- 字体 ----
    "font.family": "sans-serif",               # 可改为 "sans-serif" 或 "Times New Roman"
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,

    # ---- 轴线与边框 ----
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 1,
    "axes.edgecolor": "black",
    
    # ---- 刻度与网格 ----
    "xtick.direction": "out",
    "ytick.direction": "out",
    "xtick.major.size": 4,
    "ytick.major.size": 4,
    "xtick.major.width": 1,
    "ytick.major.width": 1,
    "grid.linestyle": "--",
    "grid.linewidth": 0.7,
    "grid.alpha": 0.4,
    
    # ---- 图例 ----
    "legend.frameon": False,
    "legend.handlelength": 1.2,
    "legend.handleheight": 0.8,
    
    # ---- 图形布局 ----
    "figure.dpi": 200,
    "figure.figsize": (6, 4),
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

In [ ]:
def add_pvalue_bracket(ax, text, x1=1, x2=1.95,
                       y_top=100.0, inset=3.0, height=2.0,
                       text_x=None, text_dy=0.6):
    """
    画一个如论文图风格的 p 值括号 (p 值文字在下方)
    ax: matplotlib Axes
    text: 例如 "p < 0.001"
    x1, x2: 括号两端位置
    y_top: 纵轴上限
    inset: 括号离上限的距离
    height: 括号高度
    """
    # 括号顶端（在 y_top 之下）
    y_base = y_top - inset
    y_top_line = y_base + height

    if text_x is None:
        text_x = (x1 + x2) / 2.0

    # 画“∪”形括号（两端竖线朝上）
    ax.plot([x1, x1, x2, x2],
            [y_base, y_top_line, y_top_line, y_base],
            lw=1.2, c="black", clip_on=False)

    # p 值文字放在括号下方
    ax.text(text_x, y_base - text_dy, text,
            ha="center", va="top", fontsize=11, clip_on=False)

In [ ]:
# ==== 加载数据 ====
und_path = "./output_csv/GoogleNQ_UND_gpt4o_Ragas.csv"
fs_path  = "./output_csv/GoogleNQ_FS_gpt4o_Ragas.csv"

UND = pd.read_csv(und_path)
FS  = pd.read_csv(fs_path)

# === 提取 EM 值并转为百分比 ===
und_vals = UND["em"].astype(float).dropna().values * 100
fs_vals  = FS["em"].astype(float).dropna().values * 100

means = [und_vals.mean(), fs_vals.mean()]
sems  = [und_vals.std()/np.sqrt(len(und_vals)),
         fs_vals.std()/np.sqrt(len(fs_vals))]

# === 绘图 ===
fig, ax = plt.subplots(figsize=(6,5))
bars = ax.bar(
    [1, 2],
    means,
    yerr=sems,
    color=["#FFD700", "#6495ED"],
    edgecolor="black",
    capsize=5,
    width=0.6
)

# === 横轴标签 ===
ax.set_xticks([1, 2])
ax.set_xticklabels(["UND", "FS"], fontsize=11)

# === 纵轴 ===
ax.set_ylabel("Exact Match (EM) Accuracy (%)", fontsize=12)
ax.set_title("Exact Match (EM) — UND vs FS", fontsize=13, weight="bold")
ax.grid(axis="y", linestyle="--", alpha=0.5)

# === 在 bar 内添加文本（百分比 + 样本数） ===
for i, (bar, mean, n) in enumerate(zip(bars, means, [len(und_vals), len(fs_vals)])):
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width()/2,
        height * 0.5,   # 放在柱子中部
        f"{mean:.1f}%\n(n={n})",
        ha="center", va="center",
        color="black" if i == 0 else "white",  # 根据颜色切换字体色
        fontsize=10, fontweight="bold"
    )

# === 显著性标注（p 值） ===
p_text = "p < 0.0001"
ymax = ax.get_ylim()[1]
ax.plot([1, 1, 1.95, 1.95], [ymax*0.95, ymax, ymax, ymax*0.95], lw=1, c="black")
ax.text(1.5, ymax*1.01, p_text, ha="center", va="bottom")

plt.tight_layout()
plt.show()

In [ ]:
# 取 F1 列并转为百分比
und_vals = UND["f1"].astype(float).dropna().values
fs_vals  = FS["f1"].astype(float).dropna().values

if und_vals.max() <= 1 and fs_vals.max() <= 1:
    und_vals *= 100
    fs_vals  *= 100

# ===============================
# 绘制 violin plot
# ===============================
fig, ax = plt.subplots(figsize=(6, 5))
p_text = "p < 0.0001"

# 为每个组定义颜色
colors = ["#FFD700", "#6495ED"]  # 金黄 / 蓝色

# 分别绘制 UND 和 FS 两个 violin，以便控制颜色
for i, (vals, color, label) in enumerate(zip(
    [und_vals, fs_vals], colors, ["UND", "FS"]
), start=1):
    parts = ax.violinplot(vals, positions=[i],
                          showmeans=False, showmedians=True, showextrema=False)
    for pc in parts['bodies']:
        pc.set_facecolor(color)
        pc.set_edgecolor('black')
        pc.set_alpha(0.6)
    if 'cmedians' in parts:
        parts['cmedians'].set_color('black')
        parts['cmedians'].set_linewidth(1.5)

# ===============================
# 中位数标注
# ===============================
medians = [np.median(und_vals), np.median(fs_vals)]
for i, (m, color) in enumerate(zip(medians, colors), start=1):
    ax.scatter(i, m, s=30, color="black", zorder=3)
    ax.text(i, m + 2.5, f"Median = {m:.1f}%", ha="center", va="bottom",
            fontsize=9, color="black", fontweight="bold")

# ===============================
# 坐标轴与标题
# ===============================
ax.set_xticks([1, 2])
ax.set_xticklabels([f"UND\n(n={len(und_vals)})", f"FS\n(n={len(fs_vals)})"], fontsize=11)
ax.set_ylabel("F1 Score (%)", fontsize=12)
ax.set_title("F1 Score — UND vs FS", fontsize=13, weight="bold", pad=30)
ax.grid(axis="y", linestyle="--", alpha=0.5)

# ===============================
# p 值标注
# ===============================
# === 固定纵轴上限为 100，并在 102–106 之间画 p 值 ===
ax.set_ylim(0, 100)
add_pvalue_bracket(ax, text=p_text, x1=1, x2=1.95,
                   y_top=102.0, inset=2.0, height=2.0,
                   text_x=1.47, text_dy=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# 取 F1 列并转为百分比
und_vals = UND["f1"].astype(float).dropna().values
fs_vals  = FS["f1"].astype(float).dropna().values

if und_vals.max() <= 1 and fs_vals.max() <= 1:
    und_vals *= 100
    fs_vals  *= 100
# ---------- 绘图 ----------
fig, ax = plt.subplots(figsize=(6, 5))
colors = ["#FFD700", "#6495ED"]  # UND/FS
groups = [und_vals, fs_vals]
labels = ["UND", "FS"]

# 分开画两个 violin，开启 showmeans（不显示 median）
for i, (vals, color) in enumerate(zip(groups, colors), start=1):
    parts = ax.violinplot(vals, positions=[i],
                          showmeans=True, showmedians=False, showextrema=False)
    for pc in parts['bodies']:
        pc.set_facecolor(color)
        pc.set_edgecolor('black')
        pc.set_alpha(0.65)
    # matplotlib自带的mean标记是白点；再加一个黑点与文本更清晰
    mean = float(np.mean(vals))
    sem  = float(np.std(vals, ddof=1) / np.sqrt(len(vals)))  # 可选：标准误
    ax.scatter(i, mean, s=34, color="black", zorder=3)
    # 可选：显示误差线（注释去掉即可）
    # ax.errorbar(i, mean, yerr=sem, fmt='none', ecolor='black', elinewidth=1.1, capsize=4, zorder=3)
    ax.text(i, min(mean + 2.5, 96.0), f"Mean = {mean:.1f}%",
            ha="center", va="bottom", fontsize=9, fontweight="bold")

# 轴与标题
ns = [len(und_vals), len(fs_vals)]
ax.set_xticks([1, 2])
ax.set_xticklabels([f"UND\n(n={ns[0]})", f"FS\n(n={ns[1]})"], fontsize=11)
ax.set_ylabel("F1 Score (%)", fontsize=12)
ax.set_title("F1 Score — UND vs FS", fontsize=13, weight="bold", pad=30)
ax.grid(axis="y", linestyle="--", alpha=0.5)

# 固定纵轴 0–100，并在其下方添加 p 值括号（右端略左移避免贴边/重叠）
ax.set_ylim(0, 100)
add_pvalue_bracket(ax, text="p < 0.0001",
                   x1=1.0, x2=1.95,
                   y_top=103.0, inset=2.8, height=1.8,
                   text_x=1.47, text_dy=0.45)
plt.tight_layout()
plt.show()

In [ ]:
# ---------- 取 ragas_AA_short 并转为百分比 ----------
col = "ragas_AA_short"
und_vals = UND[col].astype(float).dropna().values
fs_vals  = FS[col].astype(float).dropna().values

if und_vals.max() <= 1 and fs_vals.max() <= 1:
    und_vals *= 100
    fs_vals  *= 100

# ---------- 绘图 ----------
fig, ax = plt.subplots(figsize=(6, 5))
p_text = "p < 0.0001"   # ← 替换为你的真实 p 值

colors = ["#FFD700", "#6495ED"]  # UND=黄, FS=蓝
groups = [und_vals, fs_vals]
labels = ["UND", "FS"]

# 分别画两个 violin 以便独立上色
for i, (vals, color) in enumerate(zip(groups, colors), start=1):
    parts = ax.violinplot(vals, positions=[i],
                          showmeans=False, showmedians=True, showextrema=False)
    for pc in parts['bodies']:
        pc.set_facecolor(color)
        pc.set_edgecolor('black')
        pc.set_alpha(0.6)
    if 'cmedians' in parts:
        parts['cmedians'].set_color('black')
        parts['cmedians'].set_linewidth(1.5)

# 中位数点与文字
medians = [np.median(und_vals), np.median(fs_vals)]
for i, m in enumerate(medians, start=1):
    ax.scatter(i, m, s=30, color="black", zorder=3)
    ax.text(i, m + 2.5, f"Median = {m:.1f}%",
            ha="center", va="bottom", fontsize=9, fontweight="bold")

# 轴与标题
ax.set_xticks([1, 2])
ax.set_xticklabels([f"UND\n(n={len(und_vals)})", f"FS\n(n={len(fs_vals)})"], fontsize=11)
ax.set_ylabel("RAGAS Answer Accuracy (short) (%)", fontsize=12)
ax.set_title("RAGAS Answer Accuracy (short) — UND vs FS", fontsize=13, weight="bold", pad =50)
ax.grid(axis="y", linestyle="--", alpha=0.5)

# p 值括号（整体略左移避免与右侧重叠）
ax.set_ylim(0, 100)
add_pvalue_bracket(ax, text=p_text, x1=1, x2=1.95,
                   y_top=110.0, inset=2.0, height=2.0,
                   text_x=1.47, text_dy=0.2)

plt.tight_layout()
plt.show()

In [ ]:
# 取 ragas_AA_short 列并转为百分比
und_vals = UND["ragas_AA_short"].astype(float).dropna().values
fs_vals  = FS["ragas_AA_short"].astype(float).dropna().values

if und_vals.max() <= 1 and fs_vals.max() <= 1:
    und_vals *= 100
    fs_vals  *= 100

# ---------- p值括号函数（弯钩朝上、文字在下方） ----------
def add_pvalue_bracket(ax, text, x1=1.0, x2=1.95,
                       y_top=100.0, inset=4.0, height=2.0,
                       text_x=None, text_dy=0.5):
    """
    在 y_top 下方画“∪”形括号，p 值文字在括号下方。
    """
    y_base = y_top - inset
    y_top_line = y_base + height
    if text_x is None:
        text_x = (x1 + x2) / 2.0
    ax.plot([x1, x1, x2, x2],
            [y_base, y_top_line, y_top_line, y_base],
            lw=1.2, c="black", clip_on=False)
    ax.text(text_x, y_base - text_dy, text,
            ha="center", va="top", fontsize=11, clip_on=False)

# ---------- 绘图 ----------
fig, ax = plt.subplots(figsize=(6, 5))
colors = ["#FFD700", "#6495ED"]  # UND/FS
groups = [und_vals, fs_vals]
labels = ["UND", "FS"]

# 分开画两个 violin，开启 showmeans（不显示 median）
for i, (vals, color) in enumerate(zip(groups, colors), start=1):
    parts = ax.violinplot(vals, positions=[i],
                          showmeans=True, showmedians=False, showextrema=False)
    for pc in parts['bodies']:
        pc.set_facecolor(color)
        pc.set_edgecolor('black')
        pc.set_alpha(0.65)

    # 计算并标注均值
    mean = float(np.mean(vals))
    sem  = float(np.std(vals, ddof=1) / np.sqrt(len(vals)))  # 可选：标准误
    ax.scatter(i, mean, s=34, color="black", zorder=3)
    # 可选误差线
    # ax.errorbar(i, mean, yerr=sem, fmt='none', ecolor='black', elinewidth=1.1, capsize=4, zorder=3)
    ax.text(i, min(mean + 2.5, 96.0), f"Mean = {mean:.1f}%",
            ha="center", va="bottom", fontsize=9, fontweight="bold")

# 轴与标题
ns = [len(und_vals), len(fs_vals)]
ax.set_xticks([1, 2])
ax.set_xticklabels([f"UND\n(n={ns[0]})", f"FS\n(n={ns[1]})"], fontsize=11)
ax.set_ylabel("RAGAS Answer Accuracy (short) (%)", fontsize=12)
ax.set_title("RAGAS Answer Accuracy (short) — UND vs FS",
             fontsize=13, weight="bold", pad=30)
ax.grid(axis="y", linestyle="--", alpha=0.5)

# 固定纵轴 0–100，不拔高；p值放在下方留白区
ax.set_ylim(0, 100)
add_pvalue_bracket(ax, text="p < 0.0001",
                   x1=1.0, x2=1.95,
                   y_top=105.0, inset=5.0, height=1.8,
                   text_x=1.47, text_dy=0.45)

# 调整布局，让标题与图顶之间更舒展
plt.subplots_adjust(top=0.87)
plt.show()

In [ ]:
# ========= 参数与样式 =========
und_path = "./output_csv/GoogleNQ_UND_gpt4o_Ragas.csv"
fs_path  = "./output_csv/GoogleNQ_FS_gpt4o_Ragas.csv"

COLORS = ["#FFD700", "#6495ED"]  # UND / FS
LABELS = ["UND", "FS"]

# ========= 工具函数 =========
def to_percent_if_needed(a: np.ndarray) -> np.ndarray:
    """若数值最大不超过 1，则乘以 100 变为百分比。"""
    a = a.astype(float)
    if np.nanmax(a) <= 1.0:
        a = a * 100.0
    return a

def sem(a: np.ndarray) -> float:
    a = a.astype(float)
    return float(np.nanstd(a, ddof=1) / np.sqrt(np.sum(~np.isnan(a))))

def add_pvalue_bracket(ax, text, x1=1.0, x2=2.0,
                       y_top=100.0, inset=5.0, height=2.0,
                       text_x=None, text_dy=0.6, lw=1.2):
    """
    在 y_top 下方画“∪”形括号，p 值文字在括号下方（适合 0-100% 纵轴）。
    x1/x2 是两组的横坐标（与 set_xticks 对齐），y_top 是图顶参考值（可略高于 ylim 上限），
    inset 控制括号整体下移，height 控制括号高度。
    """
    y_base = y_top - inset
    y_top_line = y_base + height
    if text_x is None:
        text_x = (x1 + x2) / 2.0
    ax.plot([x1, x1, x2, x2],
            [y_base, y_top_line, y_top_line, y_base],
            lw=lw, c="black", clip_on=False)
    ax.text(text_x, y_base - text_dy, text,
            ha="center", va="top", fontsize=11, clip_on=False)

# ========= 读数 =========
UND = pd.read_csv(und_path)
FS  = pd.read_csv(fs_path)

# ========= 准备三组数据 =========
# 1) EM（已知是 0–1 或 0–100，统一转为百分比）
und_em = to_percent_if_needed(UND["em"].dropna().values)
fs_em  = to_percent_if_needed(FS["em"].dropna().values)

em_means = [float(np.nanmean(und_em)), float(np.nanmean(fs_em))]
em_sems  = [sem(und_em), sem(fs_em)]
em_ns    = [len(und_em), len(fs_em)]

# 2) F1
und_f1 = to_percent_if_needed(UND["f1"].dropna().values)
fs_f1  = to_percent_if_needed(FS["f1"].dropna().values)
f1_ns  = [len(und_f1), len(fs_f1)]

# 3) RAGAS AA short
und_aa = to_percent_if_needed(UND["ragas_AA_short"].dropna().values)
fs_aa  = to_percent_if_needed(FS["ragas_AA_short"].dropna().values)
aa_ns  = [len(und_aa), len(fs_aa)]

# ========= 画三联图 =========
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
ax0, ax1, ax2 = axes

# ---- 左：EM 柱状图 ----
bars = ax0.bar([1, 2], em_means, yerr=em_sems,
               color=COLORS, edgecolor="black", capsize=5, width=0.6)
ax0.set_xticks([1, 2])
ax0.set_xticklabels(LABELS, fontsize=11)
ax0.set_ylabel("Accuracy / Score (%)", fontsize=12)  # 与其他两图共享同一纵轴含义
ax0.set_title("Exact Match (EM)", fontsize=13, weight="bold", pad=30)
ax0.grid(axis="y", linestyle="--", alpha=0.5)

# 在柱内标注均值和样本数
for i, (bar, mean, n) in enumerate(zip(bars, em_means, em_ns)):
    height = bar.get_height()
    # 文字颜色按背景对比来简单区分（可按需微调）
    text_color = "black" if i == 0 else "white"
    ax0.text(bar.get_x() + bar.get_width()/2,
             height * 0.5,
             f"{mean:.1f}%\n(n={n})",
             ha="center", va="center",
             color=text_color, fontsize=10, fontweight="bold")

# p 值标注（放到图顶稍上方）
ax0.set_ylim(0, 100)
add_pvalue_bracket(ax0, text="p < 0.0001",
                   x1=1.0, x2=2.0,
                   y_top=40.0, inset=5.5, height=2.0,
                   text_x=1.5, text_dy=0.7)

# ---- 中：F1 小提琴图 ----
for i, (vals, color) in enumerate(zip([und_f1, fs_f1], COLORS), start=1):
    parts = ax1.violinplot(vals, positions=[i],
                           showmeans=True, showmedians=False, showextrema=False)
    for pc in parts['bodies']:
        pc.set_facecolor(color)
        pc.set_edgecolor('black')
        pc.set_alpha(0.65)
    # 均值点与标签
    m = float(np.mean(vals))
    ax1.scatter(i, m, s=34, color="black", zorder=3)
    ax1.text(i, min(m + 2.5, 96.0), f"Mean = {m:.1f}%",
             ha="center", va="bottom", fontsize=9, fontweight="bold")

ax1.set_xticks([1, 2])
ax1.set_xticklabels([f"UND\n(n={f1_ns[0]})", f"FS\n(n={f1_ns[1]})"], fontsize=11)
ax1.set_title("F1 Score", fontsize=13, weight="bold", pad=30)
ax1.grid(axis="y", linestyle="--", alpha=0.5)
ax1.set_ylim(0, 100)
add_pvalue_bracket(ax1, text="p < 0.0001",
                   x1=1.0, x2=2.0,
                   y_top=105.0, inset=5.5, height=2.0,
                   text_x=1.5, text_dy=0.7)

# ---- 右：RAGAS AA (short) 小提琴图 ----
for i, (vals, color) in enumerate(zip([und_aa, fs_aa], COLORS), start=1):
    parts = ax2.violinplot(vals, positions=[i],
                           showmeans=True, showmedians=False, showextrema=False)
    for pc in parts['bodies']:
        pc.set_facecolor(color)
        pc.set_edgecolor('black')
        pc.set_alpha(0.65)
    m = float(np.mean(vals))
    ax2.scatter(i, m, s=34, color="black", zorder=3)
    ax2.text(i, min(m + 2.5, 96.0), f"Mean = {m:.1f}%",
             ha="center", va="bottom", fontsize=9, fontweight="bold")

ax2.set_xticks([1, 2])
ax2.set_xticklabels([f"UND\n(n={aa_ns[0]})", f"FS\n(n={aa_ns[1]})"], fontsize=11)
ax2.set_title("RAGAS Answer Accuracy (short)", fontsize=13, weight="bold", pad=30)
ax2.grid(axis="y", linestyle="--", alpha=0.5)
ax2.set_ylim(0, 100)
add_pvalue_bracket(ax2, text="p < 0.0001",
                   x1=1.0, x2=2.0,
                   y_top=105.0, inset=5.5, height=2.0,
                   text_x=1.5, text_dy=0.7)

# ---- 总标题与布局 ----
fig.suptitle("UND vs FS — EM / F1 / RAGAS-AA (short)", fontsize=14, weight="bold", y=1.02)
plt.tight_layout()
plt.show()